# 06 — Sequence Alignment and BLAST with Biopython

**From real NCBI sequences to similarity analysis**

This notebook builds on Notebook 05. It covers pairwise global/local alignment, scoring parameters, a live NCBI BLAST search, parsing BLAST XML, and exporting results.

**Data:** reuse the real TP53 FASTA downloaded in Notebook 05, or download `NM_000546.6` from NCBI. The controlled alignment query is derived from that real sequence but is intentionally modified for teaching. BLAST requires internet access.

> Sequence similarity is evidence to investigate, not proof of identical function or clinical significance.


## Learning objectives

- Read real nucleotide records with `Bio.SeqIO`.
- Use `Bio.Align.PairwiseAligner` for global and local alignment.
- Explore match, mismatch, and gap scoring.
- Submit a sequence to NCBI BLAST with `Bio.Blast.NCBIWWW`.
- Parse BLAST XML with `Bio.Blast.NCBIXML`.
- Save results and metadata for reproducibility.


In [1]:
# If needed in Jupyter:
# %pip install biopython

import os
import csv
import json
import time
import hashlib
from pathlib import Path
from datetime import datetime, timezone

from Bio import SeqIO, Align
from Bio.SeqUtils import gc_fraction

print("Biopython imports loaded.")


Biopython imports loaded.


## 1. Locate data from Notebook 05

Run this notebook from the same working directory as Notebook 05 to reuse `data_ncbi/NM_000546.6.fasta`. If it is missing, Section 2 downloads the real record from NCBI.


In [2]:
DATA_DIR = Path("data_ncbi")
DATA_DIR.mkdir(exist_ok=True)
tp53_path = DATA_DIR / "NM_000546.6.fasta"

print("Working directory:", Path.cwd())
print("Existing TP53 FASTA:", tp53_path.exists())


Working directory: C:\Users\mobin\Downloads
Existing TP53 FASTA: True


## 2. Load or download a real reference sequence

NCBI Entrez requires a contact email. If the file is not already present, set `Entrez.email` to your real email before fetching the record.


In [3]:
if tp53_path.exists() and tp53_path.stat().st_size > 0:
    reference_record = SeqIO.read(tp53_path, "fasta")
    print("Loaded local file:", tp53_path)
else:
    from Bio import Entrez
    Entrez.email = os.environ.get("NCBI_EMAIL", "")
    if not Entrez.email:
        raise ValueError("Set Entrez.email = 'your.real.email@example.com' before downloading.")
    with Entrez.efetch(db="nucleotide", id="NM_000546.6",
                       rettype="fasta", retmode="text") as handle:
        fasta_text = handle.read()
    if not fasta_text.strip():
        raise RuntimeError("NCBI returned an empty FASTA response.")
    tp53_path.write_text(fasta_text, encoding="utf-8")
    reference_record = SeqIO.read(tp53_path, "fasta")

reference_seq = str(reference_record.seq).upper()
print("Accession:", reference_record.id)
print("Length:", len(reference_seq), "nt")
print("Description:", reference_record.description)


Loaded local file: data_ncbi\NM_000546.6.fasta
Accession: NM_000546.6
Length: 2512 nt
Description: NM_000546.6 Homo sapiens tumor protein p53 (TP53), transcript variant 1, mRNA


## 3. Create a controlled comparison

We derive a short segment from the real reference and introduce several substitutions plus a short deletion. The query is synthetic, but based on real NCBI sequence data; this makes the alignment effects easier to verify. It is not a second biological sample.


In [4]:
segment_start = min(100, max(0, len(reference_seq) // 10))
segment_end = min(segment_start + 180, len(reference_seq))
reference_segment = reference_seq[segment_start:segment_end]
if len(reference_segment) < 30:
    raise ValueError("Reference sequence is unexpectedly short.")

query_chars = list(reference_segment)
for pos in (12, 47, 91, 133):
    if pos < len(query_chars):
        query_chars[pos] = next(b for b in "ACGT" if b != query_chars[pos])
if len(query_chars) > 75:
    del query_chars[65:69]
query_seq = "".join(query_chars)

print("Reference segment length:", len(reference_segment))
print("Derived query length:", len(query_seq))
print("Reference:", reference_segment[:80], "...")
print("Query:    ", query_seq[:80], "...")


Reference segment length: 180
Derived query length: 176
Reference: CTTCCCTGGATTGGCAGCCAGACTGCCTTCCGGGTCACTGCCATGGAGGAGCCGCAGTCAGATCCTAGCGTCGAGCCCCC ...
Query:     CTTCCCTGGATTAGCAGCCAGACTGCCTTCCGGGTCACTGCCATGGAAGAGCCGCAGTCAGATCCGTCGAGCCCCCTCTG ...


## 4. Global alignment

Global alignment attempts to align both sequences end-to-end. Scoring parameters are explicit so the analysis is reproducible.


In [5]:
global_aligner = Align.PairwiseAligner()
global_aligner.mode = "global"
global_aligner.match_score = 2.0
global_aligner.mismatch_score = -1.0
global_aligner.open_gap_score = -5.0
global_aligner.extend_gap_score = -0.5

global_alignments = global_aligner.align(reference_segment, query_seq)
best_global = global_alignments[0]
print("Optimal alignments:", len(global_alignments))
print("Score:", best_global.score)
print(best_global)


Optimal alignments: 2
Score: 333.5
target            0 CTTCCCTGGATTGGCAGCCAGACTGCCTTCCGGGTCACTGCCATGGAGGAGCCGCAGTCA
                  0 ||||||||||||.||||||||||||||||||||||||||||||||||.||||||||||||
query             0 CTTCCCTGGATTAGCAGCCAGACTGCCTTCCGGGTCACTGCCATGGAAGAGCCGCAGTCA

target           60 GATCCTAGCGTCGAGCCCCCTCTGAGTCAGGAAACATTTTCAGACCTATGGAAACTACTT
                 60 ||||----|||||||||||||||||||||||.||||||||||||||||||||||||||||
query            60 GATC----CGTCGAGCCCCCTCTGAGTCAGGCAACATTTTCAGACCTATGGAAACTACTT

target          120 CCTGAAAACAACGTTCTGTCCCCCTTGCCGTCCCAAGCAATGGATGATTTGATGCTGTCC
                120 |||||||||||||.||||||||||||||||||||||||||||||||||||||||||||||
query           116 CCTGAAAACAACGATCTGTCCCCCTTGCCGTCCCAAGCAATGGATGATTTGATGCTGTCC

target          180 
                180 
query           176 



## 5. Local alignment

Local alignment identifies the highest-scoring subsections. It is useful when sequences overlap only in a region. Global and local scores should not be compared as though they were automatically on the same scale.


In [6]:
local_aligner = Align.PairwiseAligner()
local_aligner.mode = "local"
local_aligner.match_score = 2.0
local_aligner.mismatch_score = -1.0
local_aligner.open_gap_score = -5.0
local_aligner.extend_gap_score = -0.5

best_local = local_aligner.align(reference_segment, query_seq)[0]
print("Local score:", best_local.score)
print(best_local)
print("Coordinates:", best_local.coordinates)


Local score: 333.5
target            0 CTTCCCTGGATTGGCAGCCAGACTGCCTTCCGGGTCACTGCCATGGAGGAGCCGCAGTCA
                  0 ||||||||||||.||||||||||||||||||||||||||||||||||.||||||||||||
query             0 CTTCCCTGGATTAGCAGCCAGACTGCCTTCCGGGTCACTGCCATGGAAGAGCCGCAGTCA

target           60 GATCCTAGCGTCGAGCCCCCTCTGAGTCAGGAAACATTTTCAGACCTATGGAAACTACTT
                 60 ||||----|||||||||||||||||||||||.||||||||||||||||||||||||||||
query            60 GATC----CGTCGAGCCCCCTCTGAGTCAGGCAACATTTTCAGACCTATGGAAACTACTT

target          120 CCTGAAAACAACGTTCTGTCCCCCTTGCCGTCCCAAGCAATGGATGATTTGATGCTGTCC
                120 |||||||||||||.||||||||||||||||||||||||||||||||||||||||||||||
query           116 CCTGAAAACAACGATCTGTCCCCCTTGCCGTCCCAAGCAATGGATGATTTGATGCTGTCC

target          180 
                180 
query           176 

Coordinates: [[  0  64  68 180]
 [  0  64  64 176]]


## 6. Change gap penalties

An aligner optimizes the scoring scheme you specify. Make gaps more expensive and inspect how the score and alignment change.


In [7]:
strict_aligner = Align.PairwiseAligner()
strict_aligner.mode = "global"
strict_aligner.match_score = 2.0
strict_aligner.mismatch_score = -1.0
strict_aligner.open_gap_score = -10.0
strict_aligner.extend_gap_score = -2.0
strict_alignment = strict_aligner.align(reference_segment, query_seq)[0]

print("Default gap penalties:", best_global.score)
print("Stricter gap penalties:", strict_alignment.score)
print(strict_alignment)


Default gap penalties: 333.5
Stricter gap penalties: 324.0
target            0 CTTCCCTGGATTGGCAGCCAGACTGCCTTCCGGGTCACTGCCATGGAGGAGCCGCAGTCA
                  0 ||||||||||||.||||||||||||||||||||||||||||||||||.||||||||||||
query             0 CTTCCCTGGATTAGCAGCCAGACTGCCTTCCGGGTCACTGCCATGGAAGAGCCGCAGTCA

target           60 GATCCTAGCGTCGAGCCCCCTCTGAGTCAGGAAACATTTTCAGACCTATGGAAACTACTT
                 60 ||||----|||||||||||||||||||||||.||||||||||||||||||||||||||||
query            60 GATC----CGTCGAGCCCCCTCTGAGTCAGGCAACATTTTCAGACCTATGGAAACTACTT

target          120 CCTGAAAACAACGTTCTGTCCCCCTTGCCGTCCCAAGCAATGGATGATTTGATGCTGTCC
                120 |||||||||||||.||||||||||||||||||||||||||||||||||||||||||||||
query           116 CCTGAAAACAACGATCTGTCCCCCTTGCCGTCCCAAGCAATGGATGATTTGATGCTGTCC

target          180 
                180 
query           176 



## 7. Save an alignment summary

Record the parameters, accession, segment coordinates, and scores in JSON.


In [8]:
summary = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "reference_accession": reference_record.id,
    "segment_start_zero_based": segment_start,
    "segment_end_zero_based_exclusive": segment_end,
    "reference_segment_length": len(reference_segment),
    "derived_query_length": len(query_seq),
    "query_is_synthetic_derivative": True,
    "global_score": best_global.score,
    "local_score": best_local.score,
    "match_score": global_aligner.match_score,
    "mismatch_score": global_aligner.mismatch_score,
    "open_gap_score": global_aligner.open_gap_score,
    "extend_gap_score": global_aligner.extend_gap_score,
}
summary_path = DATA_DIR / "pairwise_alignment_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("Saved:", summary_path.resolve())


Saved: C:\Users\mobin\Downloads\data_ncbi\pairwise_alignment_summary.json


## 8. Submit a sequence to NCBI BLAST

BLAST searches a database for locally similar sequences. This live request requires internet access and may take several minutes. The example submits a short segment to the nucleotide database (`nt`). For research, choose a database and query that match your biological question, and avoid repeated unnecessary submissions.

The cell reuses a saved XML result if one exists. Delete that file only when you intentionally want to submit a new job.


In [10]:
from Bio.Blast import NCBIWWW, NCBIXML

blast_xml_path = DATA_DIR / "tp53_segment_blast.xml"

if blast_xml_path.exists() and blast_xml_path.stat().st_size > 0:
    print("Using saved BLAST result:", blast_xml_path)
else:
    result_handle = NCBIWWW.qblast(
        program="blastn",
        database="nt",
        sequence=reference_segment,
        hitlist_size=10,
        megablast=True
    )
    try:
        xml_text = result_handle.read()
    finally:
        result_handle.close()
    if not xml_text.strip():
        raise RuntimeError("BLAST returned an empty response.")
    blast_xml_path.write_text(xml_text, encoding="utf-8")
    print("Saved:", blast_xml_path.resolve())


Saved: C:\Users\mobin\Downloads\data_ncbi\tp53_segment_blast.xml


## 9. Parse BLAST XML

The E-value estimates how many matches of similar quality might be expected by chance in a database search. It depends on the search and database; interpret it alongside alignment length, identity, coverage, and biological context.


In [11]:
if not blast_xml_path.exists() or blast_xml_path.stat().st_size == 0:
    print("Run Section 8 first.")
else:
    with blast_xml_path.open(encoding="utf-8") as handle:
        blast_record = NCBIXML.read(handle)
    print("Query:", blast_record.query)
    print("Database:", blast_record.database)
    print("Alignments:", len(blast_record.alignments))
    for alignment in blast_record.alignments[:10]:
        if alignment.hsps:
            hsp = alignment.hsps[0]
            identity = 100 * hsp.identities / hsp.align_length if hsp.align_length else 0
            print({
                "hit": alignment.hit_def,
                "alignment_length": hsp.align_length,
                "identity_percent": round(identity, 2),
                "e_value": hsp.expect,
                "bit_score": hsp.bits,
            })


Query: No definition line
Database: core_nt
Alignments: 10
{'hit': 'PREDICTED: Gorilla gorilla gorilla tumor protein p53 (TP53), transcript variant X2, mRNA', 'alignment_length': 180, 'identity_percent': 100.0, 'e_value': 6.29355e-87, 'bit_score': 333.517}
{'hit': 'Homo sapiens tumor protein p53 (TP53), transcript variant 4, mRNA >gi|1894803053|ref|NM_001126113.3| Homo sapiens tumor protein p53 (TP53), transcript variant 4, mRNA', 'alignment_length': 180, 'identity_percent': 100.0, 'e_value': 6.29355e-87, 'bit_score': 333.517}
{'hit': 'Homo sapiens tumor protein p53 (TP53), transcript variant 1, mRNA >gi|1889514023|ref|NM_001276760.3| Homo sapiens tumor protein p53 (TP53), transcript variant 1, mRNA', 'alignment_length': 180, 'identity_percent': 100.0, 'e_value': 6.29355e-87, 'bit_score': 333.517}
{'hit': 'Homo sapiens cDNA FLJ54299 complete cds, highly similar to Cellular tumor antigen p53', 'alignment_length': 180, 'identity_percent': 100.0, 'e_value': 6.29355e-87, 'bit_score': 333.5

## 10. Export BLAST hits to CSV

This makes the top HSP for each hit available for spreadsheet review or downstream analysis.


In [12]:
blast_csv_path = DATA_DIR / "tp53_segment_blast_hits.csv"

if not blast_xml_path.exists() or blast_xml_path.stat().st_size == 0:
    print("Run Section 8 first.")
else:
    with blast_xml_path.open(encoding="utf-8") as handle:
        blast_record = NCBIXML.read(handle)
    rows = []
    for alignment in blast_record.alignments:
        if not alignment.hsps:
            continue
        hsp = alignment.hsps[0]
        rows.append({
            "hit_id": alignment.hit_id,
            "hit_description": alignment.hit_def,
            "alignment_length": hsp.align_length,
            "identities": hsp.identities,
            "identity_percent": round(100 * hsp.identities / hsp.align_length, 3) if hsp.align_length else 0,
            "e_value": hsp.expect,
            "bit_score": hsp.bits,
        })
    if rows:
        with blast_csv_path.open("w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=list(rows[0]))
            writer.writeheader()
            writer.writerows(rows)
        print("Saved:", blast_csv_path.resolve())
    else:
        print("No hits to export.")


Saved: C:\Users\mobin\Downloads\data_ncbi\tp53_segment_blast_hits.csv


## 11. Check file integrity

SHA-256 checksums help detect whether saved files have changed. They do not establish that a biological interpretation is correct.


In [13]:
def sha256_file(file_path):
    digest = hashlib.sha256()
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

for file_path in (tp53_path, summary_path, blast_xml_path):
    if file_path.exists() and file_path.is_file():
        print(file_path.name, sha256_file(file_path))


NM_000546.6.fasta 73a25519420ed758fa96cd8cb8904d9a6405178977ce2514e250ccf73f873005
pairwise_alignment_summary.json c5126f3338814e6cb15ca6ffc88f06e6a01728a2ae63e94e1c901bf95a7f5ef5
tp53_segment_blast.xml 8b08c268529bb0053e8d47a485f5d8d85a5bad2401ee764ed02469d46557831d


## Exercises

1. Change the scoring parameters and explain how the alignment changes.
2. Obtain two real accessions for the same gene from different organisms and align them.
3. Repeat BLAST with `megablast=False`. How do the results differ?
4. Filter hits by E-value and percent identity. Explain why both measures matter.
5. Explain why high identity alone does not prove identical function.
6. Add the BLAST program, database, query sequence, and retrieval time to the metadata.

## Common pitfalls

- Pairwise alignment works offline; NCBI download and BLAST require internet access.
- BLAST is a remote service and may be slow or temporarily unavailable.
- An empty result can reflect the query, database choice, or network/service problems.
- Parse only completed XML responses.
- Scores and E-values are not standalone biological conclusions.

## Quick reference

| Task | Biopython tool |
|---|---|
| Global/local alignment | `Align.PairwiseAligner` |
| Submit BLAST | `Bio.Blast.NCBIWWW.qblast()` |
| Parse BLAST XML | `Bio.Blast.NCBIXML.read()` |
| Export result tables | Python `csv` |
| Record metadata | Python `json` |

**Next step:** compare biologically matched sequences and build a documented sequence-comparison workflow.
